In [ ]:
include("../src/naml.jl")

In [ ]:
p = 3
prec = 20
K = PadicField(p, prec)

In [ ]:
# For testing. Creates the DAG matrix representation
# of a line with `n` nodes. Index of top node is `1`, and we go upwards.
function mk_line_matrix(n::Int64) 
    D = Matrix{Bool}(undef, n, n)
    for i in 1:n 
        for j in 1:n 
            D[i, j] = (j < i)
        end 
    end
    return D
end     

D = mk_line_matrix(10)

Here we're trying to embed a DAG in the disc space. We represent DAGs using a boolean matrix $D$, where $D_{ij}$ is true iff there is an edge $j \to i$. 

The main loss function we work with is
$$
    \mathcal{L} = \sum_{D_{ij} = 1} \log \frac{e ^ {- d(\theta_i, \theta_j)}}{\sum_{D_{ik} = 0} e ^ {- d(\theta_i, \theta_k)}}
$$
where $\theta_i$ is the polydisc embedding corresponding to node $i$. 
This is basically the same as is used in this paper: https://arxiv.org/pdf/1705.08039 (but working with the polydisc space rather than the hyperbolic disc!)

In [ ]:
# TODO: implement various operations on abstract polydisc functions and re-implement this using these operations

# "Naive" loss function, using absolute polynomials to try and capture the distance between nodes. This does not work well!
function init_rational_loss(D::Matrix{Bool}, K)
    # TODO: do we also want some kind of method for sparse evaluation?
    # For now type is hard-coded. Should be changed later on...
    linear_polynomials = Matrix{LinearPolynomial{PadicFieldElem}}(undef, size(D, 1), size(D, 2)) #([], K(0))
    for i in axes(D, 1)
        for j in axes(D, 2)
            linear_polynomials[i, j] = LinearPolynomial([u == i ? K(1) : K(0) for u in axes(D, 1)] - [u == j ? K(1) : K(0) for u in axes(D, 1)], K(0))
        end
    end
    linear_polynomials = map(batch_evaluate_init, linear_polynomials) 

    function eval_matrix(params::ValuationPolydisc{S, T})::Matrix{Float64} where S where T
        return map(f -> f(params), linear_polynomials)
    end

    # This can be vectorised!
    function eval(params::ValuationPolydisc{S, T})::Float64 where S where T
        pre_computed = eval_matrix(params)
        result = Float64(0)
        for i in axes(D, 1)
            den = Float64(0)
            num = Float64(0)
            for j in axes(D, 2)
                if !D[i, j] 
                    den += pre_computed[i, j]
                else
                    num += pre_computed[i, j]
                end
            end
            result += num / den
        end
        return result
    end
    return Loss(p -> map(eval, p), p -> 0)
end

# TODO: we may want to add a term to penalise lack of injectivity in the embedding! But maybe this is just something to be solved by initialising in a
# smarter way?

# The loss function used in the Meta paper.
function init_distance_loss(D::Matrix{Bool}, K::Ring)
    # Quick and dirty implementation for now: we'll vectorise and optimise later.

    function eval(params::ValuationPolydisc{S, T})::Float64 where S where T
        result = Float64(0)
        # TODO: implement iterate for polydiscs
        discs = components(params)
        for i in axes(D, 1)
            den = Float64(0)
            num = Float64(0)
            for j in axes(D, 2)
                if !D[i, j] 
                    den += exp(- dist(discs[i], discs[j]))
                else
                    num += exp(-dist(discs[i], discs[j]))
                end
            end
            if num == 0
                continue
            else
                result += log(num / den)
            end
        end
        return result
    end
    return Loss(p -> map(eval, p), p -> 0)
end 

## TODO: generic Gauss point constructor
function mk_initial_for(D::Matrix{Bool})
    return ValuationPolydisc(zeros(K, size(D, 1)), zeros(Int64, size(D, 1)))
end

ℓ_poly = init_rational_loss(D, K)
ℓ_dist = init_distance_loss(D, K)

next_branch = 1

# Here we don't want to force every coordinate to shrink! 
# The whole point being that which coordinates need to shrink more should be decided by the loss in this situation. 
config = (false, 1)

initial = mk_initial_for(D)

# We optimise using Greedy descent
greedy_optim_with_poly::OptimSetup = greedy_descent_init(initial, ℓ_poly, next_branch, config)

In [ ]:
N_epochs = 1000
t1 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(greedy_optim_with_poly))
    step!(greedy_optim_with_poly)
end 
t2 = time()

println("Result of optimisation process is $(greedy_optim_with_poly.param)")

In [ ]:
N_epochs = 1000
greedy_optim_with_distance::OptimSetup = greedy_descent_init(initial, ℓ_dist, next_branch, config)
t1 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(greedy_optim_with_distance))
    step!(greedy_optim_with_distance)
end 
t2 = time()

println("Result of optimisation process is $(greedy_optim_with_distance.param)")